# supra50m → Mercury-2 diffusion LM **by the translator C alone** (project method)

**Contract.** Unlike the original Mercury/DiffuLLaMA recipe (gradual adaptation of the
transformer — see the `supra_to_mercury.ipynb` BASELINE), here **model B is never trained**.
The translator **C** is a per-block, weight-tied hypernet that reads a block's
gauge-invariant weight signature and emits a **low-rank delta** for its 7 projections,
norm-gain corrections, and a [MASK] input embedding (a learned combination of the donor's
OWN embedding rows — token identities are shared, so the combination is frame-free).
C is trained through the diffusion loss on a **zoo of small AR Llamas** (depths 2–4, fresh
seeds) and then applied **zero-shot** to the 12-layer donor:  `B* = C(supra50m)`.

**Data budget.** Everything (zoo training and C training) uses only text **sampled from
supra50m itself** — the project's allowed 'Q-A captured from the model'. B* sees no data.

**Phase-10 grounding.** E1: the per-slot correction a real interior needs is small and
low-rank (rank≈16). E0/E3: a weight-tied per-block translator extrapolates over depth for
non-recursive computation — and real-LM interiors are mostly refinement (phase 9).

**Setup.** GPU + Internet, Run All (~1–1.5 h). Saves the translated checkpoint.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import glob, json, math, time, torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import load_file, save_file

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
MODEL_ID = 'SupraLabs/Supra-50M-Instruct'

def resolve_model():
    env = os.environ.get('CKPT_DIR')
    if env and os.path.exists(os.path.join(env, 'config.json')): return env
    ds = glob.glob('/kaggle/input/**/config.json', recursive=True)
    if ds: return os.path.dirname(ds[0])
    from huggingface_hub import snapshot_download
    return snapshot_download(MODEL_ID, allow_patterns=['config.json', 'model.safetensors', 'tokenizer.json'])

MODEL_DIR = resolve_model()

# --- budget knobs ---
N_GEN     = 1024     # self-generated corpus sequences (the entire data budget)
N_HELD    = 64
SEQ_LEN   = 256
ZOO_DEPTHS = [2, 3, 4, 2, 3, 4]   # six fresh small donors (depth x seed)
DONOR_STEPS = 1200
DONOR_BS  = 16
C_STEPS   = 3000     # translator training steps (through the diffusion loss)
C_BS      = 8
RANK      = 16       # emitted low-rank delta rank (phase-10 E1: rank~16 suffices)
D_Z       = 16
EPS_T     = 0.05
print('device', DEV, '| model', MODEL_DIR)

In [ ]:
tj = json.load(open(os.path.join(MODEL_DIR, 'tokenizer.json')))
inv_vocab = {i: t for t, i in tj['model']['vocab'].items()}
def decode(ids):
    return ''.join(inv_vocab.get(int(i), '?') for i in ids).replace('\u2581', ' ') \
             .replace('\u0120', ' ').replace('\u010a', '\n')

CFG = json.load(open(os.path.join(MODEL_DIR, 'config.json')))
H, KV = CFG['num_attention_heads'], CFG.get('num_key_value_heads', CFG['num_attention_heads'])
D = CFG['hidden_size']; HD = CFG.get('head_dim', D // H)
EPS = CFG.get('rms_norm_eps', 1e-5)
rp = CFG.get('rope_parameters') or {}
THETA = CFG.get('rope_theta', rp.get('rope_theta', 10000))
V = CFG['vocab_size']
MASK_ID = V                      # sentinel in input ids; outputs stay over V real tokens

def rotate_half(x):
    d = x.shape[-1] // 2
    return torch.cat([-x[..., d:], x[..., :d]], dim=-1)

def rope(x, pos):
    inv = 1.0 / (THETA ** (torch.arange(0, HD, 2, device=DEV).float() / HD))
    fr = pos[:, None].float() * inv[None, :]
    cos = torch.cat([fr.cos(), fr.cos()], -1)[None, None]
    sin = torch.cat([fr.sin(), fr.sin()], -1)[None, None]
    return x * cos + rotate_half(x) * sin

def rms(x, w): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + EPS) * w

PROJ = ('self_attn.q_proj', 'self_attn.k_proj', 'self_attn.v_proj', 'self_attn.o_proj',
        'mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj')

def llama_forward(w, idx, L, causal=True, mask_row=None):
    """Functional Llama: w = dict of weights (HF names). MASK_ID inputs use mask_row.
    Logits over the V real tokens (tied head). Differentiable wrt anything in w/mask_row."""
    B, T = idx.shape
    pos = torch.arange(T, device=DEV)
    E = w['model.embed_tokens.weight']
    x = E[idx.clamp_max(V - 1)]
    if mask_row is not None:
        x = torch.where((idx == MASK_ID)[..., None], mask_row, x)
    bias = torch.full((T, T), float('-inf'), device=DEV).triu(1)[None, None] if causal else None
    for l in range(L):
        p = lambda n: w[f'model.layers.{l}.{n}.weight']
        h = rms(x, p('input_layernorm'))
        q = (h @ p('self_attn.q_proj').T).view(B, T, H, HD).transpose(1, 2)
        k = (h @ p('self_attn.k_proj').T).view(B, T, KV, HD).transpose(1, 2)
        v = (h @ p('self_attn.v_proj').T).view(B, T, KV, HD).transpose(1, 2)
        q, k = rope(q, pos), rope(k, pos)
        k, v = k.repeat_interleave(H // KV, 1), v.repeat_interleave(H // KV, 1)
        att = (q @ k.transpose(-1, -2)) / (HD ** 0.5)
        if bias is not None: att = att + bias
        o = (att.softmax(-1) @ v).transpose(1, 2).reshape(B, T, D)
        x = x + o @ p('self_attn.o_proj').T
        h2 = rms(x, p('post_attention_layernorm'))
        x = x + (F.silu(h2 @ p('mlp.gate_proj').T) * (h2 @ p('mlp.up_proj').T)) @ p('mlp.down_proj').T
    return rms(x, w['model.norm.weight']) @ E.T

supra = {k: v.float().to(DEV) for k, v in load_file(os.path.join(MODEL_DIR, 'model.safetensors')).items()}
L_SUPRA = CFG['num_hidden_layers']
print(f'supra50m: L={L_SUPRA} d={D} V={V}')

In [ ]:
@torch.no_grad()
def generate_ar(w, L, idx, n, temperature=0.9, top_k=40):
    for _ in range(n):
        lo = llama_forward(w, idx[:, -1024:], L)[:, -1, :] / max(temperature, 1e-6)
        vv, _ = torch.topk(lo, top_k)
        lo[lo < vv[:, [-1]]] = float('-inf')
        idx = torch.cat([idx, torch.multinomial(lo.softmax(-1), 1)], 1)
    return idx

t0 = time.time()
chunks, need, gb = [], N_GEN + N_HELD, 64
temps = [0.7, 0.9, 1.0]
while sum(c.shape[0] for c in chunks) < need:
    tt = temps[len(chunks) % len(temps)]
    bos = torch.full((gb, 1), 1, dtype=torch.long, device=DEV)
    chunks.append(generate_ar(supra, L_SUPRA, bos, SEQ_LEN, temperature=tt)[:, 1:])
corpus = torch.cat(chunks, 0)[:need]
train_ids, held_ids = corpus[:N_GEN], corpus[N_GEN:]
print(f'corpus from the donor: train {tuple(train_ids.shape)} held {tuple(held_ids.shape)} in {time.time()-t0:.0f}s')

In [ ]:
def make_llama(L, seed):
    g = torch.Generator(device='cpu').manual_seed(seed)
    w = {'model.embed_tokens.weight': torch.randn(V, D, generator=g) * 0.02,
         'model.norm.weight': torch.ones(D)}
    shapes = {'self_attn.q_proj': (H*HD, D), 'self_attn.k_proj': (KV*HD, D),
              'self_attn.v_proj': (KV*HD, D), 'self_attn.o_proj': (D, H*HD),
              'mlp.gate_proj': (CFG['intermediate_size'], D), 'mlp.up_proj': (CFG['intermediate_size'], D),
              'mlp.down_proj': (D, CFG['intermediate_size'])}
    for l in range(L):
        for n, sh in shapes.items():
            w[f'model.layers.{l}.{n}.weight'] = torch.randn(*sh, generator=g) * 0.02
        w[f'model.layers.{l}.input_layernorm.weight'] = torch.ones(D)
        w[f'model.layers.{l}.post_attention_layernorm.weight'] = torch.ones(D)
    return {k: v.to(DEV).requires_grad_(True) for k, v in w.items()}

zoo = []
for di, Lz in enumerate(ZOO_DEPTHS):
    w = make_llama(Lz, seed=100 + di)
    opt = torch.optim.AdamW(list(w.values()), lr=3e-4)
    g = torch.Generator(device=DEV).manual_seed(di)
    t0 = time.time()
    for s in range(DONOR_STEPS):
        bi = torch.randint(0, train_ids.shape[0], (DONOR_BS,), device=DEV)
        x = train_ids[bi]
        lo = llama_forward(w, x[:, :-1], Lz)
        loss = F.cross_entropy(lo.reshape(-1, V), x[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(list(w.values()), 1.0)
        opt.step()
    w = {k: v.detach() for k, v in w.items()}
    zoo.append((w, Lz))
    print(f'donor {di} (L={Lz}): final AR CE {loss.item():.2f}  ({time.time()-t0:.0f}s)')

In [ ]:
def sample_mask_rate(b, eps=EPS_T, device=DEV):
    return eps + (1.0 - eps) * torch.rand(b, device=device)

def forward_mask(x0, t):
    B, L = x0.shape
    noise = torch.rand(B, L, device=x0.device)
    m = noise < t[:, None].expand(B, L)
    empty = ~m.any(dim=1)
    if empty.any():
        m[empty.nonzero(as_tuple=True)[0], noise[empty].argmin(dim=1)] = True
    return torch.where(m, torch.full_like(x0, MASK_ID), x0), m

def diffusion_loss(logits, x0, m, t):
    B, L, Vv = logits.shape
    ce = F.cross_entropy(logits.reshape(-1, Vv), x0.reshape(-1), reduction='none').view(B, L)
    return ((ce * m).sum(dim=1) / (t * L)).mean()

@torch.no_grad()
def denoise(forward_fn, ids, frozen, steps=64, temperature=0.0):
    B, L = ids.shape
    for s in range(steps):
        masked = (ids == MASK_ID) & ~frozen
        n_left = int(masked.sum().item())
        if n_left == 0: break
        logits = forward_fn(ids)
        probs = (logits / temperature).softmax(-1) if temperature > 0 else logits.softmax(-1)
        pred = torch.multinomial(probs.view(-1, V), 1).view(B, L) if temperature > 0 else probs.argmax(-1)
        conf = probs.max(-1).values.masked_fill(~masked, float('-inf'))
        k = min(max(1, n_left // (steps - s)), n_left)
        reveal = conf.view(-1).topk(k).indices
        ids.view(-1)[reveal] = pred.view(-1)[reveal]
    masked = (ids == MASK_ID) & ~frozen
    if masked.any():
        ids = torch.where(masked, forward_fn(ids).argmax(-1), ids)
    return ids

@torch.no_grad()
def masked_ce_at(forward_fn, ids, t_val, n=32):
    x0 = ids[:n]
    x_t, m = forward_mask(x0, torch.full((x0.shape[0],), t_val, device=DEV))
    lo = forward_fn(x_t)
    ce = F.cross_entropy(lo.reshape(-1, V), x0.reshape(-1), reduction='none').view(x0.shape)
    return (ce * m).sum().item() / m.sum().item()
print('diffusion core ready')

In [ ]:
def block_signature(w, l):
    """Gauge-invariant per-block descriptor: top-32 normalized singular values of each of
    the 7 projections + their log-norms. Cached per frozen donor (weights never change)."""
    parts, logn = [], []
    for n in PROJ:
        W = w[f'model.layers.{l}.{n}.weight']
        s = torch.linalg.svdvals(W.float())
        parts.append((s / (s.norm() + 1e-9))[:32])
        logn.append(torch.log(W.norm() + 1e-9)[None])
    return torch.cat(parts + logn)                          # 7*32 + 7 = 231 dims

SIG_DIM = 7 * 32 + 7
OUT_SHAPES = {'self_attn.q_proj': (H*HD, D), 'self_attn.k_proj': (KV*HD, D),
              'self_attn.v_proj': (KV*HD, D), 'self_attn.o_proj': (D, H*HD),
              'mlp.gate_proj': (CFG['intermediate_size'], D), 'mlp.up_proj': (CFG['intermediate_size'], D),
              'mlp.down_proj': (D, CFG['intermediate_size'])}

class TranslatorC(nn.Module):
    """Per-block weight-tied hypernet: signature(+depth_frac) -> z -> low-rank deltas for
    the 7 projections + norm-gain corrections; plus a global frame-free [MASK] embedding
    (softmax combination over the donor's OWN embedding rows). Deltas init 0 => B starts
    exactly at the raw-bidirectional donor (the floor)."""
    def __init__(self, d_z=D_Z, r=RANK, h=128):
        super().__init__()
        self.r = r
        self.enc = nn.Sequential(nn.Linear(SIG_DIM + 1, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, d_z))
        self.MU = nn.ParameterDict(); self.MV = nn.ParameterDict()
        for n, (o, i) in OUT_SHAPES.items():
            key = n.replace('.', '_')
            self.MU[key] = nn.Parameter(torch.randn(o * r, d_z) * 0.01)
            self.MV[key] = nn.Parameter(torch.zeros(r * i, d_z))    # zero => delta starts 0
        self.Mg = nn.Parameter(torch.zeros(2 * D, d_z))             # norm-gain corrections
        self.mask_logits = nn.Parameter(torch.zeros(V))             # frame-free [MASK] combo

    def mask_row(self, w):
        return self.mask_logits.softmax(0) @ w['model.embed_tokens.weight']

    def emit(self, w, L, sigs):
        """Return overlay weight dict for an L-layer donor (donor tensors stay frozen)."""
        out = dict(w)
        for l in range(L):
            frac = 0.0 if L == 1 else l / (L - 1)
            z = self.enc(torch.cat([sigs[l], sigs[l].new_tensor([frac])]))
            for n, (o, i) in OUT_SHAPES.items():
                key = n.replace('.', '_')
                U = (self.MU[key] @ z).view(o, self.r)
                Vt = (self.MV[key] @ z).view(self.r, i)
                out[f'model.layers.{l}.{n}.weight'] = w[f'model.layers.{l}.{n}.weight'] + U @ Vt
            dg = (self.Mg @ z).view(2, D)
            out[f'model.layers.{l}.input_layernorm.weight'] = \
                w[f'model.layers.{l}.input_layernorm.weight'] * (1 + dg[0])
            out[f'model.layers.{l}.post_attention_layernorm.weight'] = \
                w[f'model.layers.{l}.post_attention_layernorm.weight'] * (1 + dg[1])
        return out

C = TranslatorC().to(DEV)
with torch.no_grad():
    zoo_sigs = [[block_signature(w, l).to(DEV) for l in range(Lz)] for w, Lz in zoo]
    supra_sigs = [block_signature(supra, l).to(DEV) for l in range(L_SUPRA)]
print('C params:', sum(p.numel() for p in C.parameters()))

In [ ]:
opt = torch.optim.AdamW(C.parameters(), lr=3e-4)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 100))
ema, t0 = None, time.time()
for step in range(1, C_STEPS + 1):
    di = step % len(zoo)
    w, Lz = zoo[di]
    bi = torch.randint(0, train_ids.shape[0], (C_BS,), device=DEV)
    x0 = train_ids[bi]
    t = sample_mask_rate(C_BS)
    x_t, m = forward_mask(x0, t)
    wB = C.emit(w, Lz, zoo_sigs[di])
    lo = llama_forward(wB, x_t, Lz, causal=False, mask_row=C.mask_row(w))
    loss = diffusion_loss(lo, x0, m, t)
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(C.parameters(), 1.0)
    opt.step(); sched.step()
    ema = loss.item() if ema is None else 0.98 * ema + 0.02 * loss.item()
    if step % 500 == 0 or step == 1:
        with torch.no_grad():
            w0, L0 = zoo[0]
            fn = lambda i: llama_forward(C.emit(w0, L0, zoo_sigs[0]), i, L0, causal=False,
                                         mask_row=C.mask_row(w0))
            hce = masked_ce_at(fn, held_ids, 0.5, n=16)
        print(f'step {step:>5}  train-ELBO(ema) {ema:6.3f}  zoo-donor0 held masked-CE@0.5 {hce:5.2f}'
              f'  ({time.time()-t0:.0f}s)')
print('C trained — B was never trained, only C was')

In [ ]:
with torch.no_grad():
    wB = C.emit(supra, L_SUPRA, supra_sigs)            # B* = C(supra50m), zero-shot
    mrow = C.mask_row(supra)
fnB  = lambda i: llama_forward(wB, i, L_SUPRA, causal=False, mask_row=mrow)
# floor: raw supra run bidirectionally with a mean-embedding mask row, NO translation
mean_row = supra['model.embed_tokens.weight'].mean(0)
fn0  = lambda i: llama_forward(supra, i, L_SUPRA, causal=False, mask_row=mean_row)

print('uniform ln(V) = %.2f | donor AR next-token CE (held) = %.2f' % (math.log(V),
      F.cross_entropy(llama_forward(supra, held_ids[:16, :-1], L_SUPRA).reshape(-1, V),
                      held_ids[:16, 1:].reshape(-1)).item()))
print('\nheld masked-CE:   floor(raw bidir)  vs  B*=C(supra)   [lower=better]')
for tv in (0.3, 0.5, 0.7, 0.9):
    print(f'  t={tv}:  {masked_ce_at(fn0, held_ids, tv):6.2f}        {masked_ce_at(fnB, held_ids, tv):6.2f}')

# reconstruction of 25%-masked held text (16 parallel steps)
x0 = held_ids[:8]
x_c, m = forward_mask(x0, torch.full((x0.shape[0],), 0.25, device=DEV))
for name, fn in (('floor', fn0), ('B*=C(supra)', fnB)):
    rec = denoise(fn, x_c.clone(), ~m, steps=16)
    acc = ((rec == x0) & m).sum().item() / m.sum().item()
    print(f'\nreconstruction ({name}): token accuracy {acc:.1%}')
    if name.startswith('B'):
        print('  original :', repr(decode(x0[0, :48])))
        print('  recovered:', repr(decode(rec[0, :48])))

# parallel generation from scratch (the Mercury mode), via B* only
ids = torch.full((2, 128), MASK_ID, dtype=torch.long, device=DEV)
frozen = torch.zeros(2, 128, dtype=torch.bool, device=DEV)
gen = denoise(fnB, ids, frozen, steps=64, temperature=0.7)
for b in range(2):
    print(f'\nB* parallel sample {b}:', repr(decode(gen[b])[:300]))

# control: signatures matter? emit supra with SHUFFLED block signatures
with torch.no_grad():
    perm = torch.randperm(L_SUPRA).tolist()
    wB_mm = C.emit(supra, L_SUPRA, [supra_sigs[p] for p in perm])
fn_mm = lambda i: llama_forward(wB_mm, i, L_SUPRA, causal=False, mask_row=mrow)
print(f'\ncontrol — shuffled-signature emit, masked-CE@0.5: {masked_ce_at(fn_mm, held_ids, 0.5):.2f} '
      f'(vs B* {masked_ce_at(fnB, held_ids, 0.5):.2f}; if equal, C ignores signatures)')

In [ ]:
out_sd = {k: v.detach().cpu().contiguous() for k, v in wB.items()}
out_sd['mercury.mask_embedding'] = mrow.detach().cpu().contiguous()
path = '/kaggle/working/supra_mercury2_by_C.safetensors' if os.path.isdir('/kaggle/working') \
       else 'supra_mercury2_by_C.safetensors'
save_file(out_sd, path)
print('saved B* = C(supra50m) ->', path, f'({os.path.getsize(path)/1e6:.0f} MB)')

## How to read

- **Floor vs B\*** masked-CE: the floor is the untranslated donor forced bidirectional —
  if `B* = C(supra)` beats it decisively across mask rates, the translator alone moved the
  model toward a working denoiser **with zero training of B**. Compare against the BASELINE
  notebook's post-finetune numbers (the original-paper ceiling) to see what fraction of the
  floor→ceiling gap C closes by reading weights.
- **Shuffled-signature control**: if it matches B\*, C learned a signature-independent
  (per-depth-only) rule; a gap means C genuinely reads the block's weights.
- **Honest expectations**: C trains on depth-2..4 fresh donors and extrapolates to a
  well-trained L=12 model — per phase-10 this works for refinement-style interiors but the
  early unique layers may be under-served; partial gap closure is already the headline.
  This notebook never fine-tunes B: every improvement over the floor is C's doing.